# Phase 1 C-MAPSS EDA

This notebook is a thin, reproducible front end over the package code. It expects the NASA C-MAPSS text files to exist locally and keeps generated outputs under `artifacts/phase1`.

In [ ]:
from pathlib import Path

import pandas as pd

from aerospace_prognostics.analysis.cmapss_eda import build_cmapss_eda_report
from aerospace_prognostics.data.cmapss import CMAPSS_SUBSETS, load_cmapss_subset
from aerospace_prognostics.data.manifest import build_cmapss_manifest, verify_manifest
from aerospace_prognostics.evaluation import write_results_csv, write_results_json
from aerospace_prognostics.experiments.cmapss_baseline import run_all_cmapss_hist_gradient_boosting

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data" / "raw" / "cmapss"
ARTIFACT_DIR = PROJECT_ROOT / "artifacts" / "phase1"
RUL_CAP = 125
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

## Dataset Provenance

Build a local manifest, verify checksums immediately, and write the manifest as a Phase 1 artifact.

In [ ]:
manifest = build_cmapss_manifest(DATA_DIR)
problems = verify_manifest(manifest, root=DATA_DIR)
if problems:
    raise ValueError(problems)

manifest_path = ARTIFACT_DIR / "data" / "cmapss_manifest.json"
manifest.write_json(manifest_path)

pd.DataFrame([entry.__dict__ for entry in manifest.entries])

## EDA Summaries

Write one JSON report per FD subset and keep a compact table in the notebook for inspection.

In [ ]:
eda_rows = []
eda_dir = ARTIFACT_DIR / "eda"
for subset in CMAPSS_SUBSETS:
    bundle = load_cmapss_subset(DATA_DIR, subset, rul_cap=RUL_CAP)
    report = build_cmapss_eda_report(bundle)
    report.write_json(eda_dir / f"{subset.lower()}.json")
    eda_rows.append(
        {
            "subset": subset,
            "train_units": report.train_units,
            "test_units": report.test_units,
            "train_rows": report.train_rows,
            "test_rows": report.test_rows,
            "flat_sensors": len(report.flat_sensors),
        }
    )

pd.DataFrame(eda_rows)

## Classical Baseline

Run the Phase 1 gradient-boosted baseline across all subsets and persist JSON plus CSV results.

In [ ]:
baseline_results = run_all_cmapss_hist_gradient_boosting(
    DATA_DIR,
    rul_cap=RUL_CAP,
    random_state=42,
    standardize=True,
)
results_dir = ARTIFACT_DIR / "results"
write_results_json(baseline_results, results_dir / "cmapss_baseline.json")
write_results_csv(baseline_results, results_dir / "cmapss_baseline.csv")

pd.DataFrame([result.to_dict() for result in baseline_results])

## One-Command Workflow

The notebook cells above are useful for inspection. For repeatable runs, prefer the CLI workflow:

```powershell
uv run aerospace-prognostics phase1-cmapss --data-dir data/raw/cmapss --artifact-dir artifacts/phase1
```